# M09 — Make Binary Decisions

**Objective:** build a binary classifier and reason about probabilities and thresholds.

Whole-first route:

**baseline → split → classifier → predicted probabilities → default classification → confusion matrix → threshold changes → consequences**

This lab is CPU-only, offline, deterministic, and uses synthetic data. It produces practice observations but does not prefill learner evidence.


## 1. See the complete decision system

The **model** maps features to an estimated probability. The **decision policy** compares that probability with a threshold. Evaluation compares the resulting class with the binary target.

Before running anything, predict which stages can change when only the threshold changes. Record that prediction outside the notebook output.


In [ ]:
from pathlib import Path
import csv
import math
import random
from statistics import fmean

FEATURES = [
    "account_age_days",
    "weekly_sessions",
    "overdue_tasks",
    "assessment_score",
    "help_requests",
]
TARGET = "disengaged_next_30_days"

def locate_repository_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "datasets" / "M09" / "learner_disengagement.csv").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the LearningOS-AI repository")

ROOT = locate_repository_root()
DATASET = ROOT / "datasets" / "M09" / "learner_disengagement.csv"

with DATASET.open(encoding="utf-8", newline="") as handle:
    raw_rows = list(csv.DictReader(handle))

rows = []
for raw in raw_rows:
    row = {"learner_id": raw["learner_id"]}
    for name in FEATURES + [TARGET]:
        row[name] = int(raw[name])
    rows.append(row)

print(f"loaded {len(rows)} synthetic rows from {DATASET.relative_to(ROOT)}")
print("features:", FEATURES)
print("target:", TARGET)

assert len(rows) == 180
assert {row[TARGET] for row in rows} == {0, 1}


## 2. Binary target and majority baseline

`1` means the synthetic learner disengages in the following 30 days; `0` means the learner does not.

**Predict before running:** If a baseline predicts `0` for every row, what are TP, TN, FP and FN? Why might its accuracy look respectable even though it never identifies a positive case?


In [ ]:
target_counts = {
    0: sum(row[TARGET] == 0 for row in rows),
    1: sum(row[TARGET] == 1 for row in rows),
}
positive_rate = target_counts[1] / len(rows)

majority_class = max(target_counts, key=target_counts.get)
baseline_confusion_all_rows = {
    "tn": target_counts[0],
    "fp": 0,
    "fn": target_counts[1],
    "tp": 0,
}
baseline_accuracy_all_rows = target_counts[majority_class] / len(rows)

print("target counts:", target_counts)
print(f"positive rate: {positive_rate:.3f}")
print("all-negative baseline confusion:", baseline_confusion_all_rows)
print(f"all-negative baseline accuracy: {baseline_accuracy_all_rows:.3f}")

assert target_counts == {0: 136, 1: 44}
assert majority_class == 0
assert baseline_confusion_all_rows["fn"] > 0


## 3. Split before fitting

We preserve a holdout and stratify by the binary target so both classes are represented. IDs, not just row counts, prove that no example crosses the boundary.

**Predict before running:** Approximately how many positives should appear in a 25% holdout when the full dataset contains 44 positives?


In [ ]:
def stratified_split(input_rows, test_fraction=0.25, seed=909):
    grouped = {0: [], 1: []}
    for row in input_rows:
        grouped[row[TARGET]].append(row)

    rng = random.Random(seed)
    train_rows = []
    test_rows = []
    for label in (0, 1):
        group = list(grouped[label])
        rng.shuffle(group)
        test_count = round(len(group) * test_fraction)
        test_rows.extend(group[:test_count])
        train_rows.extend(group[test_count:])

    rng.shuffle(train_rows)
    rng.shuffle(test_rows)
    return train_rows, test_rows

train_rows, test_rows = stratified_split(rows)
train_ids = {row["learner_id"] for row in train_rows}
test_ids = {row["learner_id"] for row in test_rows}
train_rate = fmean(row[TARGET] for row in train_rows)
test_rate = fmean(row[TARGET] for row in test_rows)

print("train size / positive rate:", len(train_rows), round(train_rate, 3))
print("test size / positive rate: ", len(test_rows), round(test_rate, 3))
print("overlapping IDs:", train_ids & test_ids)

assert len(train_rows) == 135
assert len(test_rows) == 45
assert not (train_ids & test_ids)
assert train_ids | test_ids == {row["learner_id"] for row in rows}
assert sum(row[TARGET] for row in test_rows) == 11


## 4. Fit a probability-producing classifier

The classifier is logistic regression implemented with standard-library math and deterministic batch gradient descent. Feature means and scales are learned from training rows only.

The sigmoid maps any real-valued score into `(0, 1)`. That number is interpreted as estimated positive-class probability, not as a guaranteed outcome.

**Predict before running:** Which features do you expect to receive positive or negative coefficients? Direction is more important than exact magnitude.


In [ ]:
def fit_standardizer(input_rows, feature_names):
    means = [fmean(row[name] for row in input_rows) for name in feature_names]
    scales = []
    for name, mean in zip(feature_names, means):
        variance = fmean((row[name] - mean) ** 2 for row in input_rows)
        scales.append(math.sqrt(variance) or 1.0)
    return means, scales

def transform_rows(input_rows, feature_names, means, scales):
    return [
        [
            (row[name] - mean) / scale
            for name, mean, scale in zip(feature_names, means, scales)
        ]
        for row in input_rows
    ]

feature_means, feature_scales = fit_standardizer(train_rows, FEATURES)
X_train = transform_rows(train_rows, FEATURES, feature_means, feature_scales)
y_train = [row[TARGET] for row in train_rows]
X_test = transform_rows(test_rows, FEATURES, feature_means, feature_scales)
y_test = [row[TARGET] for row in test_rows]

assert len(X_train) == len(y_train) == 135
assert len(X_test) == len(y_test) == 45


In [ ]:
def sigmoid(value):
    clipped = max(-35.0, min(35.0, value))
    return 1.0 / (1.0 + math.exp(-clipped))

def fit_logistic_classifier(X, y, epochs=1500, learning_rate=0.08, l2=0.01):
    weights = [0.0] * (len(X[0]) + 1)  # intercept, then one weight per feature

    for _ in range(epochs):
        gradient = [0.0] * len(weights)
        for features, target in zip(X, y):
            score = weights[0] + sum(
                weight * value for weight, value in zip(weights[1:], features)
            )
            error = sigmoid(score) - target
            gradient[0] += error
            for index, value in enumerate(features, start=1):
                gradient[index] += error * value

        for index in range(len(weights)):
            regularization = 0.0 if index == 0 else l2 * weights[index]
            weights[index] -= learning_rate * (
                gradient[index] / len(y) + regularization
            )

    return weights

def predict_probabilities(X, weights):
    return [
        sigmoid(weights[0] + sum(w * value for w, value in zip(weights[1:], row)))
        for row in X
    ]

model_weights = fit_logistic_classifier(X_train, y_train)
coefficient_table = [
    ("intercept", model_weights[0]),
    *zip(FEATURES, model_weights[1:]),
]

for name, value in coefficient_table:
    print(f"{name:>22}: {value: .3f}")

assert len(model_weights) == len(FEATURES) + 1
assert all(math.isfinite(value) for value in model_weights)


## 5. Inspect probabilities before classes

**Predict before running:** Will every positive target receive a probability above every negative target? Explain why noisy, overlapping features make perfect ordering unlikely.

The threshold has not appeared yet. We can rank risk and inspect uncertainty without making a binary action.


In [ ]:
test_probabilities = predict_probabilities(X_test, model_weights)
probability_records = [
    {
        "learner_id": row["learner_id"],
        "probability": probability,
        "actual": actual,
    }
    for row, probability, actual in zip(test_rows, test_probabilities, y_test)
]

print("probability range:", round(min(test_probabilities), 3), "to", round(max(test_probabilities), 3))

assert len(test_probabilities) == len(test_rows)
assert all(0.0 < probability < 1.0 for probability in test_probabilities)
assert len({round(probability, 6) for probability in test_probabilities}) > 35


In [ ]:
ranked = sorted(
    probability_records,
    key=lambda record: record["probability"],
    reverse=True,
)

print("highest estimated probabilities")
for record in ranked[:8]:
    print(
        record["learner_id"],
        f"p={record['probability']:.3f}",
        f"actual={record['actual']}",
    )

print("lowest estimated probabilities")
for record in ranked[-5:]:
    print(
        record["learner_id"],
        f"p={record['probability']:.3f}",
        f"actual={record['actual']}",
    )


## 6. Default classification at threshold 0.50

A threshold converts probability into a predicted class:

`predicted class = 1 if probability >= threshold else 0`

**Predict before running:** Count the displayed high-probability rows that will be positive at `0.50`. Would moving the threshold change the learned weights or the probabilities?


In [ ]:
def classify(probabilities, threshold=0.50):
    return [int(probability >= threshold) for probability in probabilities]

default_threshold = 0.50
default_predictions = classify(test_probabilities, default_threshold)

print("default threshold:", default_threshold)
print("predicted positives:", sum(default_predictions))

assert set(default_predictions) <= {0, 1}
assert sum(default_predictions) == 4


## 7. Confusion matrix before summary metrics

For positive class “will disengage”:

- **TP:** predicts disengagement and disengagement occurs;
- **TN:** predicts no disengagement and none occurs;
- **FP:** predicts disengagement but none occurs (possibly unnecessary outreach);
- **FN:** predicts no disengagement but disengagement occurs (a missed outreach opportunity).

**Predict before running:** Which error becomes more common when the threshold is high?


In [ ]:
def confusion_counts(actual, predicted):
    if len(actual) != len(predicted):
        raise ValueError("actual and predicted must have equal length")
    return {
        "tn": sum(a == 0 and p == 0 for a, p in zip(actual, predicted)),
        "fp": sum(a == 0 and p == 1 for a, p in zip(actual, predicted)),
        "fn": sum(a == 1 and p == 0 for a, p in zip(actual, predicted)),
        "tp": sum(a == 1 and p == 1 for a, p in zip(actual, predicted)),
    }

def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

def metric_summary(counts):
    total = sum(counts.values())
    return {
        "accuracy": safe_divide(counts["tp"] + counts["tn"], total),
        "precision": safe_divide(counts["tp"], counts["tp"] + counts["fp"]),
        "recall": safe_divide(counts["tp"], counts["tp"] + counts["fn"]),
    }

default_counts = confusion_counts(y_test, default_predictions)
default_metrics = metric_summary(default_counts)

print("confusion at 0.50 (TN, FP, FN, TP):", default_counts)
for name, value in default_metrics.items():
    print(f"{name:>9}: {value:.3f}")

assert default_counts == {"tn": 33, "fp": 1, "fn": 8, "tp": 3}
assert sum(default_counts.values()) == len(y_test)


In [ ]:
baseline_predictions = [0] * len(y_test)
baseline_counts = confusion_counts(y_test, baseline_predictions)
baseline_metrics = metric_summary(baseline_counts)

print("holdout baseline confusion:", baseline_counts)
print("holdout baseline metrics:", {name: round(value, 3) for name, value in baseline_metrics.items()})

assert baseline_counts == {"tn": 34, "fp": 0, "fn": 11, "tp": 0}
assert baseline_metrics["accuracy"] > 0.75
assert baseline_metrics["recall"] == 0.0


## 8. Change thresholds, not the model

**Predict before running:** As the threshold moves from `0.70` down to `0.20`, which directions should predicted positives, recall, and usually precision move? Record the prediction before viewing the table.

Every row below uses the same holdout targets and the same probabilities.


In [ ]:
candidate_thresholds = [0.20, 0.30, 0.50, 0.70]
threshold_results = []

for threshold in candidate_thresholds:
    predictions = classify(test_probabilities, threshold)
    counts = confusion_counts(y_test, predictions)
    metrics = metric_summary(counts)
    threshold_results.append(
        {
            "threshold": threshold,
            "predicted_positive": sum(predictions),
            **counts,
            **metrics,
        }
    )

header = "threshold predicted+  TN FP FN TP  accuracy precision recall"
print(header)
for result in threshold_results:
    print(
        f"{result['threshold']:>8.2f}"
        f"{result['predicted_positive']:>11}"
        f"{result['tn']:>4}{result['fp']:>3}{result['fn']:>3}{result['tp']:>3}"
        f"{result['accuracy']:>10.3f}"
        f"{result['precision']:>10.3f}"
        f"{result['recall']:>7.3f}"
    )

assert [row["predicted_positive"] for row in threshold_results] == [16, 10, 4, 2]
assert threshold_results[0]["recall"] > threshold_results[2]["recall"]


In [ ]:
low_threshold_predictions = classify(test_probabilities, 0.20)
changed_cases = [
    {
        "learner_id": row["learner_id"],
        "probability": probability,
        "actual": actual,
        "at_0.20": low_prediction,
        "at_0.50": default_prediction,
    }
    for row, probability, actual, low_prediction, default_prediction in zip(
        test_rows,
        test_probabilities,
        y_test,
        low_threshold_predictions,
        default_predictions,
    )
    if low_prediction != default_prediction
]

for record in sorted(changed_cases, key=lambda item: item["probability"], reverse=True):
    print(record)

assert len(changed_cases) == 12
assert all(0.20 <= record["probability"] < 0.50 for record in changed_cases)


## 9. Controlled failure: default threshold plus accuracy only

The seeded failure accepts `0.50` merely because it is conventional and reports only accuracy.

Before running the next cells, read `missions/M09/controlled_failure.md` and predict which policy will be cheaper if an FN costs five times an FP. Do not assume that the most accurate row must be cheapest.


In [ ]:
result_by_threshold = {
    result["threshold"]: result for result in threshold_results
}

accuracy_at_030 = result_by_threshold[0.30]["accuracy"]
accuracy_at_050 = result_by_threshold[0.50]["accuracy"]
recall_at_030 = result_by_threshold[0.30]["recall"]
recall_at_050 = result_by_threshold[0.50]["recall"]

print(f"0.30 accuracy={accuracy_at_030:.3f}, recall={recall_at_030:.3f}")
print(f"0.50 accuracy={accuracy_at_050:.3f}, recall={recall_at_050:.3f}")
print("same accuracy, different missed-positive behavior:", accuracy_at_030 == accuracy_at_050)

assert accuracy_at_030 == accuracy_at_050
assert recall_at_030 > recall_at_050


## 10. Make consequences explicit

For this exercise only, assign cost `1` to an unnecessary outreach (FP) and cost `5` to a missed disengagement (FN). Correct classifications have zero cost.

**Predict before running:** Which candidate threshold minimizes `1 × FP + 5 × FN`? What trade-off will it accept?


In [ ]:
FALSE_POSITIVE_COST = 1
FALSE_NEGATIVE_COST = 5

for result in threshold_results:
    result["consequence_cost"] = (
        FALSE_POSITIVE_COST * result["fp"]
        + FALSE_NEGATIVE_COST * result["fn"]
    )

for result in threshold_results:
    print(
        f"threshold={result['threshold']:.2f}",
        f"FP={result['fp']}",
        f"FN={result['fn']}",
        f"cost={result['consequence_cost']}",
    )

selected_policy = min(
    threshold_results,
    key=lambda result: (result["consequence_cost"], -result["threshold"]),
)
print("lowest-cost candidate:", selected_policy["threshold"])

assert selected_policy["threshold"] == 0.20
assert selected_policy["consequence_cost"] < result_by_threshold[0.50].get(
    "consequence_cost", float("inf")
)


Explain the selected policy in plain language:

- Which kind of error did it reduce?
- Which kind did it accept more often?
- Why is this a consequence-dependent choice rather than a universal best threshold?
- What would happen if outreach capacity were capped below its number of predicted positives?

Do not treat the toy cost values as real organizational policy.


## 11. Calibration intuition

If predictions around `0.30` are well calibrated, roughly 30% of comparable cases should be positive over many observations. Calibration is a group-level agreement idea, not a promise about one learner.

**Predict before running:** Which problem will appear when a probability bin contains only a few held-out rows?


In [ ]:
def calibration_bins(probabilities, actual, edges=(0.0, 0.2, 0.4, 0.6, 0.8, 1.000001)):
    bins = []
    for lower, upper in zip(edges, edges[1:]):
        members = [
            (probability, target)
            for probability, target in zip(probabilities, actual)
            if lower <= probability < upper
        ]
        if members:
            bins.append(
                {
                    "range": f"[{lower:.1f}, {min(upper, 1.0):.1f})",
                    "count": len(members),
                    "mean_probability": fmean(item[0] for item in members),
                    "observed_rate": fmean(item[1] for item in members),
                }
            )
    return bins

calibration_table = calibration_bins(test_probabilities, y_test)
print("range       n  mean predicted  observed positive rate")
for row in calibration_table:
    print(
        f"{row['range']:<11}{row['count']:>3}"
        f"{row['mean_probability']:>16.3f}"
        f"{row['observed_rate']:>24.3f}"
    )

assert sum(row["count"] for row in calibration_table) == len(y_test)
assert any(row["count"] < 5 for row in calibration_table)


In [ ]:
def brier_score(probabilities, actual):
    return fmean((probability - target) ** 2 for probability, target in zip(probabilities, actual))

model_brier = brier_score(test_probabilities, y_test)
constant_probability = fmean(y_train)
constant_brier = brier_score([constant_probability] * len(y_test), y_test)

print(f"model Brier score:    {model_brier:.3f}")
print(f"constant Brier score: {constant_brier:.3f}")
print("lower is better, but one small holdout is not deployment evidence")

assert 0.0 <= model_brier <= 1.0
assert model_brier < constant_brier


## 12. Verify the model/policy boundary

Changing a threshold changed predicted labels, confusion counts, metrics, and consequence cost. It did **not** refit feature scaling, coefficients, or probabilities.

Use `missions/M09/code_reading.md` to trace one case through that boundary. Then explain why a threshold is neither a learned probability nor an observed target.


In [ ]:
weights_snapshot = tuple(model_weights)
probabilities_snapshot = tuple(test_probabilities)

policy_outputs = {
    threshold: tuple(classify(test_probabilities, threshold))
    for threshold in candidate_thresholds
}

assert tuple(model_weights) == weights_snapshot
assert tuple(test_probabilities) == probabilities_snapshot
assert len(set(policy_outputs.values())) == len(candidate_thresholds)

print("one model fit;", len(candidate_thresholds), "distinct threshold policies")


## 13. No-AI gate

Complete `missions/M09/no_ai_gate.md` without AI-generated analysis or code.

It provides fresh batch probabilities, an inspection-capacity limit, asymmetric FP/FN costs, and outcomes revealed only after you commit to a threshold. Passing requires a probability-to-action decision, correct hand-computed confusion counts and metrics, and a calibration explanation.


## 14. Evidence and reflection

Submit only learner-produced evidence required by `missions/M09/evidence_contract.yaml`.

Your explanation should answer:

1. Why did the majority baseline look better than its recall?
2. What did the classifier learn, and what did the threshold decide?
3. How did each threshold change TP, TN, FP and FN?
4. Why could equal accuracy conceal different consequences?
5. What assumptions made the selected threshold reasonable only for this exercise?
6. What does calibration mean, and why are these holdout bins weak evidence?

M09 is complete only when those claims are supported by a trace or calculation and transfer to the fresh no-AI scenario.
